# Phase 1: local dataset audit

Inspect manually supplied sources with shared streaming validators. No downloads, installations, model inference or training. Synthetic tests are separate from the real source reports. Large full scans are opt-in.


In [ ]:
from pathlib import Path
import json, sys
WORKSPACE = Path.cwd().resolve()
if not (WORKSPACE / "ARCHITECTURE.md").is_file():
    WORKSPACE = WORKSPACE.parent
assert (WORKSPACE / "ARCHITECTURE.md").is_file(), "Start from project/notebooks directory"
sys.path.insert(0, str(WORKSPACE))
from dataset_tools.inventory import inventory_raw
from dataset_tools.movielens import audit_movielens
from dataset_tools.imdb import audit_imdb
from dataset_tools.resources import preflight
print(json.dumps(preflight(WORKSPACE), indent=2))


## Inventory and existing evidence

Inventory reads metadata in bounded directory batches. A file count does not establish provenance or labels. Historical audit documents describe earlier snapshots; current machine-readable reports under artifacts/data_quality record later runs.


In [ ]:
inventory = inventory_raw(WORKSPACE / "data/raw")
print(json.dumps(inventory["counts"], indent=2))


## Explicitly opted-in scans

Set RUN_FULL_AUDIT=True only when resource checks pass and a fresh output directory is selected. These calls inspect real local data and never fabricate missing inputs. Existing reports are never overwritten.


In [ ]:
RUN_FULL_AUDIT = False
if RUN_FULL_AUDIT:
    import uuid
    resources = preflight(WORKSPACE)
    assert not resources["blockers"], resources["blockers"]
    run = WORKSPACE / "artifacts/data_quality" / ("notebook-movielens-" + uuid.uuid4().hex[:8])
    result = audit_movielens(WORKSPACE / "data/raw/movielens/ml-32m", output=run)
    print(json.dumps(result, indent=2))
else:
    print("Full scans disabled. Read the saved real validation reports or opt in explicitly.")


## Interpretation

Keep ratings separate from sentiment. Missing/duplicated external IDs require quarantine, not guessed joins. IMDb remains a binary benchmark and its official test is preserved. Near-duplicate retrieval is approximate and cannot prove semantic uniqueness. Human labels, consent/provenance and approved canonical movie mappings remain independent gates.


In [ ]:
reports = sorted((WORKSPACE / "artifacts/data_quality").glob("**/report.json"))
reports += sorted((WORKSPACE / "artifacts/data_quality").glob("**/imdb_audit.json"))
for p in reports:
    report = json.loads(p.read_text(encoding="utf-8"))
    print(p.relative_to(WORKSPACE), report.get("status"), report.get("scope"), report.get("counts"))
